In [ ]:
import pypsa
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
from pypsa.plot import add_legend_lines, add_legend_patches, add_legend_semicircles
import yaml
from pathlib import Path
import pandas as pd
import yaml
from clusters.add_renewables_cluster import *


**Set Up**

In [ ]:
fn = 'resources/Iberic5_test/networks/base_s_5__12h_2050.nc'


In [ ]:
n= pypsa.Network(fn)

config = yaml.safe_load(Path("config/config.iberic5.yaml").read_text())


In [ ]:
p = Path(fn)  
try:
   if p.exists():
       p.unlink()
       print(f"Deleted {p}")
   else:
       print(f"File not found: {p}")
except Exception as e:
    print(f"Failed to delete {p}: {e}")

**Options**

In [ ]:
ongrid=False
cluster_cost_reduction=1
cluster_size=1000   
renewables={"solar",'solar-hsat','onwind'}

In [ ]:
nodes_with_clusters = n.buses.loc[
    n.buses.index.str[:2].isin(config['countries']) &
    (n.buses['carrier'] == 'AC')
].index.tolist()




**Buses and Generators of the Cluster Addition**

In [ ]:
n= assign_cluster_generators_and_electricity_buses(n, config, cluster_size, cluster_cost_reduction, renewables, nodes_with_clusters)


**Links of the Cluster Addition**

In [ ]:
n = add_cluster_links(n, nodes_with_clusters, cluster_cost_reduction, ongrid)


**Storages of the Cluster Addition**

In [ ]:
n = add_cluster_storages(n, nodes_with_clusters, cluster_cost_reduction)

In [ ]:
n.links["reversed"] = n.links["reversed"].fillna(False).astype(bool)


**Printing to Check**

**Exporting**

In [ ]:
n.export_to_netcdf(fn)
